# E. coli fed-batch: frozen-model rate reconciliation
Same source equations and operating histories; see README for the scope of validation.


### Neutral kinetic correction

The kinetic input files expose `parameters.corrections.theta = 0` (dimensionless).
The correction multiplies the selected substrate-dependent rate by
`exp(theta * substrate / (substrate + reference_concentration))`; zero preserves
the original rate. Each declaration explicitly binds the concentration and its
units. The ordinary kinetic examples reuse that rate's half-saturation
concentration as the reference scale, a configurable assumption rather than a
calibrated value. Doyle's `hybrid.json` retains its separate documented scale.
A shared theta changes all declared corrections in that file; derived yields
and balance expressions continue to use the corrected rates. Change individual
bindings to use independent coefficients if required. Selecting another kinetic
relationship also requires reviewing its input slots and correction bindings.

Pathway examples now connect empirical uptake capacities through
`model.pathways.rule_graph` and the existing `rule-graph` provider, with
`parameters.corrections.theta = 0`. Edit the capacity expression's `relationship`
and inputs to select a kinetic form; constant capacities use a parameter node.
The correction scale is the Monod affinity, or an explicit configurable
`parameters.corrections.reference_concentration = 1` [mmol/L] for constant
capacities. These are normalization assumptions, not fitted values. Monod's
`greater-select.threshold` preserves the native depletion cutoff; update it
alongside `model.pathways.depletion_tolerance`. Negative trial concentrations
use the existing zero extension. The oxygen `transfer-cap`, inventory limits,
and metabolic balances remain independent constraints, even for nonzero theta.


## Kinetics selection

Edit the named form in the input JSON, then rerun from Step 2. The processing
layer resolves the name through the existing registry; parameters, units,
stoichiometry and material destinations remain explicit. `kinetic_forms` in
Step 2 lists the available equations, aliases and required inputs. See the
example README for the exact rule identifiers and parameter paths.

Select `relationship` within `model.kinetics.rule_graph`, such as `monod`
or `haldane`. Haldane additionally requires an explicit `inhibition_constant`
input and parameter. Retain conversion, yield, inhibition and hybrid correction
nodes surrounding the selected form. Changing a form changes the scientific
model and requires new calibration/validation; it does not preserve a textbook
or experimental reproduction automatically.


In [ ]:
# STEP 1: Select runs and folders
# WHAT TO DO: Use Run All or Shift+Enter in order. Copy the input folders to configure
# another process; change INPUT_DIRS below. No measurements are needed to simulate.
from pathlib import Path
import json
import csv
import numpy as np
import matplotlib.pyplot as plt
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'PharmaPy').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from PharmaPy.Bioreactors import relationship_catalog, build_bioreactor, BioreactorUnitConverter as Units

EXAMPLE = ROOT / 'examples/bioreactors/ecoli_ye_fed_batch'
INPUT_DIRS = [EXAMPLE / 'reconciliation/inputs', EXAMPLE / 'reconciliation/inputs/F11']
THERMO = EXAMPLE / 'inputs/thermo.json'
EXPORT_DIR = EXAMPLE / 'reconciliation/outputs'
WRITE_ARTIFACTS = True


In [ ]:
# STEP 2: Load process configurations
# WHAT TO DO: Configure equations and initial amounts in mechanism.json; operation and numerical select
# runtime and numerics. Quantities declare value/unit; legacy recipe suffixes remain supported.
# See README for concentration and rate declarations. Each feed appears once.
configurations = [json.loads((p / 'mechanism.json').read_text()) for p in INPUT_DIRS]
thermo = json.loads(THERMO.read_text())

kinetic_forms = relationship_catalog()
print("Monod inputs:", kinetic_forms["saturating-uptake"]["required_inputs"])
for selected_config in configurations:
    print("Kinetic rules:", [rule["identifier"] for rule in selected_config["model"]["kinetics"]["rule_graph"]])


In [ ]:
# STEP 3: Build native reactors from the configured inputs
# WHAT TO DO: No process-specific Python is needed. Configure kinetics and recipes in JSON.
# build_bioreactor validates inputs and attaches the shared native mechanisms and streams.
assemblies = {config['operation']['recipe']['name']: build_bioreactor(config, thermo_path=THERMO)
              for config in configurations}


In [ ]:
# STEP 4: Forward simulate through native solve_unit
# WHAT TO DO: Run each assembly. The shared workflow applies flow changes, additions and
# well-mixed samples at scheduled boundaries. Rebuild in Step 3 before rerunning a simulation.
solutions = {}
for run, assembly in assemblies.items():
    histories = assembly.solve(verbose=False)
    solutions[run] = (assembly.phase, histories)
    print(run, type(assembly.unit).__name__, len(histories), 'segments completed')


In [ ]:
# STEP 5: Tabulate and plot trajectories
# WHAT TO DO: All concentrations come from native inventories divided by native working
# volume. Duplicate boundary times preserve pre/post event states. Edit plotting labels here.
tables, figures = {}, {}
for run, (phase, histories) in solutions.items():
    rows = []
    for result in histories:
        for t, mass, vol in zip(result.time, result.mass_j_liquid0, np.asarray(result.vessel_vol).reshape(-1)):
            row = {'time_h': Units.convert(t, 's', 'h'), 'volume_l': Units.convert(vol, 'm3', 'L')}
            row.update({n+'_g_l': Units.convert(m/vol, 'kg/m3', 'g/L') for n, m in zip(phase.name_species, mass)})
            row.update({n+'_g': Units.convert(m, 'kg', 'g') for n, m in zip(phase.name_species, mass)})
            rows.append(row)
    tables[run] = rows
    quantities = ['volume_l'] + [n+'_g_l' for n in phase.name_species if n != assemblies[run].carrier_species]
    fig, axes = plt.subplots(2, int(np.ceil(len(quantities)/2)), figsize=(13,7), constrained_layout=True)
    for ax, name in zip(axes.flat, quantities):
        ax.plot([r['time_h'] for r in rows], [r[name] for r in rows])
        ax.set(xlabel='Time (h)', ylabel=name); ax.grid(alpha=.25)
    fig.suptitle(run+' — native PharmaPy, single-counted feed')
    figures[run] = fig
    print(run, 'final volume (L):', rows[-1]['volume_l'])
plt.show()


In [ ]:
# STEP 6: Export forward trajectories and figures
# WHAT TO DO: Run to write CSV spreadsheets and PNG/SVG figures. Set WRITE_ARTIFACTS=False
# in Step 1 for a calculation without replacing files.
if WRITE_ARTIFACTS:
    for run, rows in tables.items():
        folder = EXPORT_DIR / run
        folder.mkdir(parents=True, exist_ok=True)
        with (folder/'trajectories.csv').open('w') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator='\n')
            writer.writeheader(); writer.writerows(rows)
        figures[run].savefig(folder/'trajectories.png', dpi=160)
        figures[run].savefig(folder/'trajectories.svg')
